# Producing basis for Quack reconstruction

QUACK produces a reconstruction by representing any electric field as a linear combination of Fourier modes. The observation effect of each electric field mode is simulated through the angular streaking simulation and that library of observations is used when fitting the reconstruction. This notebook shows how this bases can be produced and saved into an HDF5 file, which is used in QUACK.

## Theory

This shall produce a simulation of the eTOF observation, which is proportional to $|b_{i_{E,FEL}}|^2$, where the indices $i$ relate to the FEL energy. The matrix elements $\textbf{b}$ can be calculated by the time-evolution of the initial wavefunction of the photo-electrons. We assume we can apply first order perturbation theory and the wavefunction of the photo-electrons immediately after ionization is given by the dipole approximation, in which the $x$-axis FEL electric field interacts with the atom and the resulting wavefunction is $\vec{E} \cdot \vec{d}$, where $\vec{E}$ is the FEL field and $\vec{d}$ is the electric dipole in the eTOF gas. This may be produced at any point in time during the interaction with the streaking vector potential $\vec{A}$. Additionally, the photo-electrons wavefunction evolve in time. To find out how this is observed, we apply the unitary time-evolution operator $\exp\left(-\int \frac{i}{\hbar} \hat{H} dt\right)$ to the initial wavefunction.

The resulting probability of detecting a photo-electron in the eTOF at a given momentum $\textbf{p}$ is given by the following equation (see Amini, Kasra, et al. "Symphony on Strong Field Approximation." Reports on Progress in Physics, vol. 82, no. 11, Oct. 2019, p. 116001. Crossref, https://doi.org/10.1088/1361-6633/ab2bb1).


## Computational aspects

The simulation is done in Julia and hence the following cell loads some necessary elements. It is recommended to do the simulation using GPUs, as this is much faster. To enable or disable GPUs, switch the flag `use_GPU` in the options below.


In [1]:
# this is needed to load the Julia interface internally

import envmodules
envmodules.load('exfel', 'julia')
import os
os.environ["JULIA_NUM_THREADS"] = 'auto'
os.environ['PYTHON_JULIAPKG_OFFLINE'] = 'no'


- EXFEL modulepath enabled


In [2]:
from quacksim import *

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
[Tue, 09 Sep 2025 14:49:43] Loading file '/gpfs/exfel/data/scratch/danilo/envs/quack/lib/python3.10/site-packages/quacksim/src/../data/pcs_Kr1s.txt' with data for Kr4p_multi...
[Tue, 09 Sep 2025 14:49:43] Loading file '/gpfs/exfel/data/scratch/danilo/envs/quack/lib/python3.10/site-packages/quacksim/src/../data/pcs_Xe.txt' with data for Xe3d...
[Tue, 09 Sep 2025 14:49:43] Loading file '/gpfs/exfel/data/scratch/danilo/envs/quack/lib/python3.10/site-packages/quacksim/src/../data/pcs_Kr1s.txt' with data for Kr1s...
[Tue, 09 Sep 2025 14:49:43] Loading file '/gpfs/exfel/data/scratch/danilo/envs/quack/lib/python3.10/site-packages/quacksim/src/../data/pcs_Ne.txt' with data for Ne1s...
[Tue, 09 Sep 2025 14:49:43] Loading file '/gpfs/exfel/data/scratch/danilo/envs/quack/lib/python3.10/site-packages/quacksim/src/../data/pcs_N.txt' with data for N1s...
[Tue, 09 Sep 2025 14:49:43] Loadi

In [3]:
import numpy as np

Below, several settings are set, which must be adapted to the experiment at hand.

In [4]:
# photo-electron kinetic energy axis (in eV) and angular axis (in rad)
minW = 95.0
maxW = 155.0
nbinsW = 160
W_axis = np.linspace(minW, maxW, nbinsW)
theta_axis = np.arange(0, 2*np.pi, step=2*np.pi/16)

In [5]:
# whether to use GPUs
use_gpu = True

# output H5 file
output = "../example/basis.h5"

# ellipticity
epsilon = 1.0   

# gas
gas = "Ne1s"

# lser wavelength in nm
laser_wavelength = 1030


# maximum ponderomotive potentil, in eV
maxUp = 0.4

# linear FEL polarization
polarization="horizontal"

# beta: Set to 0 for circular polarization, or 2 for linear polarization
beta = 2.0

# how much to oversample the FEL energy axis
oversampling = 1

# number of bins in time and in the ponderomotive potential axis
nbins_time = 128
nbins_Up = 64

# FEL central energy in eV
fel_energy = 1000.0

In [6]:
simulate_amplitude(
                   fel_energy=fel_energy,
                   W_axis=W_axis,
                   theta_axis=theta_axis,
                   laser_wavelength=laser_wavelength,
                   maxUp=maxUp,
                   output_filename=output,
                   gas=gas,
                   nbins_time=nbins_time,
                   nbins_Up=nbins_Up,
                   oversampling=oversampling,
                   use_gpu=use_gpu,
                   beta=beta,
                   polarization=polarization,
                   ellipticity=epsilon,
                   ) 

[Tue, 09 Sep 2025 14:50:20] Using analytical result for ϕ(t=Tl)...
[Tue, 09 Sep 2025 14:50:28] Calculating b(t=Tl)...
[Tue, 09 Sep 2025 14:51:09] Progress:   5 % (@ t =   0.179 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:10] Progress:  15 % (@ t =   0.517 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:10] Progress:  25 % (@ t =   0.862 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:11] Progress:  35 % (@ t =   1.210 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:11] Progress:  45 % (@ t =   1.548 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:11] Progress:  55 % (@ t =   1.892 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:12] Progress:  65 % (@ t =   2.239 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:12] Progress:  75 % (@ t =   2.578 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:13] Progress:  85 % (@ t =   2.924 fs, span:   0.000 fs to   3.436 fs)
[Tue, 09 Sep 2025 14:51:13]